# Advanced 02 — GraphRAG: Relationship Retrieval and Provenance

**Scenario:** Acme's architecture and compliance team must explain how projects, services, vendors, controls, regions, and regulations are connected—without inventing paths, crossing tenant boundaries, or treating a derived graph as the source of truth.  
**Runtime:** deterministic and credential-free by default; optional structured extraction and answer generation use a configured model.  
**Central lesson:**

> GraphRAG is useful when explicit relationships are part of the retrieval problem.

Graph-based retrieval can improve these query classes **when the graph accurately represents the relevant entities and relationships**. It complements rather than replaces lexical, vector, or hybrid retrieval.

We preserve the original Atlas → service → supplier → regulation storyline, then evolve it from a three-edge shortest-path demo into a complete local relationship-retrieval lab:

`source text → typed extraction → validation → entity resolution → MultiDiGraph → bounded directional paths → source spans → cited answer → evaluation`


## Learning objectives, success criteria, and boundaries

By the end, you can:

- represent sources, entities, relations, paths, and query resolutions with typed contracts;
- validate extraction records before graph ingestion;
- explain and measure entity-resolution false merges and false splits;
- preserve edge-to-span-to-document/version provenance in a `networkx.MultiDiGraph`;
- retrieve directional, relation-aware, tenant-safe paths under hop and fact budgets;
- demonstrate why undirected and unweighted shortest paths can be semantically wrong;
- seed graph retrieval from text evidence when a query lacks exact graph names;
- evaluate extraction, resolution, path correctness, provenance, freshness, and authorization separately; and
- compare graph retrieval with a text-only baseline on the same labelled queries.

**Success criteria:** every generation-eligible edge has valid provenance; ambiguous mentions require clarification; current queries exclude historical facts; unauthorized edges never enter a path; traversal budgets are asserted; citations resolve to source spans.

**Non-goals:** this is not Microsoft GraphRAG's complete indexing pipeline, not a Neo4j deployment, not a universal entity resolver, and not evidence that graphs outperform text retrieval for every question.

**Prerequisites:** [Retrieval Strategies](../../intermediate/01-retrieval-strategies/README.md), [Metadata and Permissions](../../intermediate/02-metadata-permissions/README.md), [Evaluation](../../intermediate/04-evaluation/README.md), and [Corrective RAG](../01-corrective-rag/README.md).


## Executable architecture

![A source-backed multi-hop path connects Project Atlas through VectorDB-X and Acme Systems to Regulation R-17.](assets/graph-path.svg)

```text
source documents → text units → entity/relation extraction → entity resolution
       → validated property graph → query seeding → bounded traversal
       → supporting source spans → grounded cited answer
```

The graph is **derived data**. Every edge must remain retractable and auditable through its extraction record and source version.


## 1. Environment and reproducibility

The default path uses committed synthetic sources and frozen extraction fixtures. Set `GRAPHRAG_USE_LIVE_EXTRACTION=1`, `OPENAI_API_KEY`, and `GRAPHRAG_MODEL` only to experiment with structured live extraction. Set `GRAPHRAG_USE_LIVE_GENERATOR=1` to opt into live synthesis. Neither live path is required for the lab or CI.


In [ ]:
from __future__ import annotations

from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from datetime import date
import heapq
import itertools
import math
import os
import re
from typing import Literal

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
from IPython.display import display
from pydantic import BaseModel, Field
from rank_bm25 import BM25Okapi

print({
    "networkx": nx.__version__,
    "live_extraction_requested": os.getenv("GRAPHRAG_USE_LIVE_EXTRACTION") == "1",
    "live_generation_requested": os.getenv("GRAPHRAG_USE_LIVE_GENERATOR") == "1",
})


## 2. Typed source, entity, relation, query, path, and evidence contracts

The original notebook's manual `Fact(subject, predicate, object, source)` is a useful mental model. We retain it but make every boundary explicit. A relation is not authoritative merely because it became an edge.

![A graph fact contains subject, controlled relation, object, and provenance.](assets/fact-anatomy.svg)


In [ ]:
EntityType = Literal["Project", "Service", "Database", "Vendor", "BusinessUnit", "Control", "Regulation", "Owner", "Region", "Platform"]
RelationType = Literal[
    "DEPENDS_ON", "SUPPLIED_BY", "OWNS", "IMPLEMENTS_CONTROL",
    "GOVERNED_BY", "LOCATED_IN", "OPERATED_BY", "USES_PLATFORM", "HOSTS",
]


class SourceRecord(BaseModel):
    source_id: str
    document_id: str
    text: str
    tenant_id: str
    version: str
    effective_date: str | None = None
    status: Literal["current", "historical"] = "current"
    authority: Literal["official_record", "regulator", "approved_vendor", "supporting_marketing"] = "official_record"


class EntityRecord(BaseModel):
    entity_id: str
    canonical_name: str
    entity_type: EntityType
    aliases: list[str] = Field(default_factory=list)
    tenant_id: str


class RelationRecord(BaseModel):
    relation_id: str
    source_entity_id: str
    relation_type: RelationType
    target_entity_id: str
    source_id: str
    source_span: str
    source_version: str
    tenant_id: str
    status: Literal["current", "historical"] = "current"
    authority: str
    extraction_status: Literal["frozen_fixture", "live_extraction", "human_verified"] = "frozen_fixture"


class QueryEntity(BaseModel):
    mention: str
    resolved_entity_id: str | None = None
    candidate_entity_ids: list[str] = Field(default_factory=list)
    status: Literal["resolved", "ambiguous", "unresolved"]


class PathEdge(BaseModel):
    relation_id: str
    source_entity_id: str
    relation_type: str
    target_entity_id: str
    source_id: str
    source_span: str
    source_version: str
    traversal_direction: Literal["forward", "reverse"] = "forward"


class TraversalTrace(BaseModel):
    nodes_visited: int = 0
    edges_evaluated: int = 0
    facts_returned: int = 0
    max_depth: int = 0
    stopped_by_budget: bool = False


class GraphPath(BaseModel):
    seed_entity_ids: list[str]
    target_entity_id: str | None
    edges: list[PathEdge] = Field(default_factory=list)
    terminal_reason: str
    trace: TraversalTrace


class EvidenceRecord(BaseModel):
    evidence_id: str
    relation_id: str
    source_id: str
    document_id: str
    source_version: str
    source_span: str
    statement: str


class GroundedAnswer(BaseModel):
    text: str
    citation_ids: list[str]


@dataclass(frozen=True)
class Principal:
    principal_id: str
    tenant_id: str


## 3. Controlled relation vocabulary and directional schema

Direction and endpoint types are schema, not prompt suggestions. `Project DEPENDS_ON Service` is permitted; `Regulation DEPENDS_ON Project` is rejected. Reverse traversal, when a query needs it, is an explicit policy action rather than a silent conversion to an undirected graph.


In [ ]:
RELATION_SCHEMA: dict[str, tuple[set[str], set[str]]] = {
    "DEPENDS_ON": ({"Project", "Service"}, {"Service", "Database"}),
    "SUPPLIED_BY": ({"Service", "Database"}, {"Vendor"}),
    "OWNS": ({"BusinessUnit"}, {"Project", "Service"}),
    "IMPLEMENTS_CONTROL": ({"Service"}, {"Control"}),
    "GOVERNED_BY": ({"Vendor", "Control", "Platform"}, {"Regulation"}),
    "LOCATED_IN": ({"Service", "Database"}, {"Region"}),
    "OPERATED_BY": ({"Project", "Service"}, {"Owner"}),
    "USES_PLATFORM": ({"Project"}, {"Platform"}),
    "HOSTS": ({"Platform"}, {"Service"}),
}

RELATION_COST = {
    "DEPENDS_ON": 1.0,
    "SUPPLIED_BY": 1.0,
    "GOVERNED_BY": 1.0,
    "IMPLEMENTS_CONTROL": 1.1,
    "LOCATED_IN": 1.0,
    "OPERATED_BY": 1.0,
    "OWNS": 1.0,
    "USES_PLATFORM": 5.0,   # generic infrastructure context: valid, but weak for supplier questions
    "HOSTS": 4.0,           # high-degree relation: expensive and often noisy
}

GENERATION_AUTHORITIES = frozenset({"official_record", "regulator", "approved_vendor"})
MAX_HOPS = 3
MAX_FACTS = 12


## 4. Build 26 synthetic source records

The corpus keeps the Atlas/supplier/regulation scenario and adds projects, applications, services, databases, vendors, business units, controls, owners, regions, historical facts, aliases, a high-degree platform hub, and a separate Globex tenant. No secret or proprietary data is present.


In [ ]:
def S(source_id, document_id, text, tenant="acme", version="1", effective="2026-01-01", status="current", authority="official_record"):
    return SourceRecord(
        source_id=source_id, document_id=document_id, text=text, tenant_id=tenant,
        version=version, effective_date=effective, status=status, authority=authority,
    )


sources = [
    S("src-atlas-arch-v2", "atlas-architecture", "Project Atlas depends on VectorDB-X for relationship search. Project Atlas is operated by the Data Platform Team.", version="2"),
    S("src-atlas-arch-v1", "atlas-architecture", "Project Atlas depended on LegacyGraph in 2024.", version="1", effective="2024-01-01", status="historical"),
    S("src-vectordb-vendor", "approved-vendor-register", "VectorDB-X is supplied by Acme Systems Inc.", version="3"),
    S("src-vendor-marketing", "vendor-marketing-brief", "VectorDB-X is supplied by Acme Systems Inc.", version="1", authority="supporting_marketing"),
    S("src-r17-register", "regulatory-register-r17", "Acme Systems Inc. is governed by Regulation R-17.", version="4", authority="regulator"),
    S("src-vectordb-controls", "vectordb-control-record", "VectorDB-X implements the Encryption at Rest control and is located in Canada.", version="5"),
    S("src-encryption-r17", "control-mapping-r17", "The Encryption at Rest control is governed by Regulation R-17.", authority="regulator"),
    S("src-atlas-platform", "atlas-platform-map", "Project Atlas uses the Enterprise Platform.", version="2"),
    S("src-platform-r17", "platform-compliance-map", "The Enterprise Platform is governed by Regulation R-17.", authority="regulator"),
    S("src-atlas-search", "atlas-search-service", "Atlas Search Service is operated by the Search Reliability Team and implements the Access Logging control."),
    S("src-access-r22", "control-mapping-r22", "The Access Logging control is governed by Regulation R-22.", authority="regulator"),
    S("src-borealis", "borealis-architecture", "Project Borealis depends on Analytics Service."),
    S("src-analytics-vendor", "analytics-vendor-register", "Analytics Service is supplied by Northwind Data."),
    S("src-northwind-r22", "northwind-compliance", "Northwind Data is governed by Regulation R-22.", authority="regulator"),
    S("src-payments", "payments-architecture", "Checkout Modernization depends on Payments API. Payments API depends on OrdersDB and is operated by the Commerce Operations Team."),
    S("src-orders-region", "orders-database-inventory", "OrdersDB is located in the European Union."),
    S("src-risk", "risk-engine-architecture", "Risk Engine depends on Identity Service and is operated by the Risk Engineering Team."),
    S("src-identity-vendor", "identity-vendor-register", "Identity Service is supplied by Contoso Identity."),
    S("src-contoso-r17", "contoso-compliance", "Contoso Identity is governed by Regulation R-17.", authority="regulator"),
    S("src-shared-auth-ownership", "business-service-ownership", "Platform Engineering owns Shared Auth Service. Risk Business Unit owns Shared Auth Service."),
    S("src-business-ownership", "business-project-ownership", "Data Business Unit owns Project Atlas. Commerce Business Unit owns Payments API."),
    S("src-platform-hosting-a", "enterprise-platform-catalog-a", "Enterprise Platform hosts VectorDB-X, Analytics Service, and Payments API."),
    S("src-platform-hosting-b", "enterprise-platform-catalog-b", "Enterprise Platform hosts Risk Engine, Identity Service, Atlas Search Service, and Shared Auth Service."),
    S("src-globex-atlas", "globex-atlas-architecture", "Globex Project Atlas depends on Globex Vector Store.", tenant="globex"),
    S("src-globex-vendor", "globex-vendor-register", "Globex Vector Store is supplied by Acme Systems.", tenant="globex"),
    S("src-globex-r22", "globex-compliance", "Acme Systems is governed by Regulation R-22 for Globex workloads.", tenant="globex", authority="regulator"),
]

source_by_id = {source.source_id: source for source in sources}
assert 20 <= len(sources) <= 35
assert len(source_by_id) == len(sources)
display(pd.DataFrame([s.model_dump() for s in sources])[['source_id', 'tenant_id', 'version', 'status', 'authority']].head(10))
print("source records:", len(sources))


## 5. Authoritative entity registry and frozen extraction fixture

The model may propose mentions and relations, but it does not get to invent enterprise IDs. The default fixture represents reviewed extraction output. A live extractor can propose typed names and spans; deterministic resolution and validation still gate graph ingestion.


In [ ]:
def E(entity_id, name, entity_type, aliases=(), tenant="acme"):
    return EntityRecord(entity_id=entity_id, canonical_name=name, entity_type=entity_type, aliases=list(aliases), tenant_id=tenant)


entities = [
    E("project-atlas", "Project Atlas", "Project", ("Atlas",)),
    E("project-borealis", "Project Borealis", "Project", ("Borealis",)),
    E("project-checkout", "Checkout Modernization", "Project", ("Checkout Project",)),
    E("service-vectordb-x", "VectorDB-X", "Service", ("Vector DB X",)),
    E("service-vectordb-y", "VectorDB-Y", "Service"),
    E("service-legacy-graph", "LegacyGraph", "Service"),
    E("service-atlas-search", "Atlas Search Service", "Service", ("Atlas Search", "Atlas")),
    E("service-analytics", "Analytics Service", "Service"),
    E("service-payments", "Payments API", "Service"),
    E("service-risk", "Risk Engine", "Service"),
    E("service-identity", "Identity Service", "Service"),
    E("service-shared-auth", "Shared Auth Service", "Service"),
    E("database-orders", "OrdersDB", "Database"),
    E("vendor-acme", "Acme Systems", "Vendor", ("Acme Systems Inc.", "ACME")),
    E("vendor-northwind", "Northwind Data", "Vendor"),
    E("vendor-contoso", "Contoso Identity", "Vendor"),
    E("bu-data", "Data Business Unit", "BusinessUnit"),
    E("bu-commerce", "Commerce Business Unit", "BusinessUnit"),
    E("bu-risk", "Risk Business Unit", "BusinessUnit"),
    E("bu-platform", "Platform Engineering", "BusinessUnit"),
    E("control-encryption", "Encryption at Rest", "Control"),
    E("control-access", "Access Logging", "Control"),
    E("reg-r17", "Regulation R-17", "Regulation", ("R-17",), "public"),
    E("reg-r22", "Regulation R-22", "Regulation", ("R-22",), "public"),
    E("owner-data", "Data Platform Team", "Owner"),
    E("owner-search", "Search Reliability Team", "Owner"),
    E("owner-commerce", "Commerce Operations Team", "Owner"),
    E("owner-risk", "Risk Engineering Team", "Owner"),
    E("region-canada", "Canada", "Region"),
    E("region-eu", "European Union", "Region", ("EU",)),
    E("platform-enterprise", "Enterprise Platform", "Platform"),
    E("globex-project-atlas", "Globex Project Atlas", "Project", ("Project Atlas", "Atlas"), "globex"),
    E("globex-vector-store", "Globex Vector Store", "Service", tenant="globex"),
    E("globex-vendor-acme", "Acme Systems", "Vendor", ("ACME",), "globex"),
]
entity_by_id = {entity.entity_id: entity for entity in entities}


def R(relation_id, source_entity, relation_type, target_entity, source_id, source_span, extraction_status="frozen_fixture"):
    source = source_by_id[source_id]
    return RelationRecord(
        relation_id=relation_id, source_entity_id=source_entity, relation_type=relation_type,
        target_entity_id=target_entity, source_id=source_id, source_span=source_span,
        source_version=source.version, tenant_id=source.tenant_id, status=source.status,
        authority=source.authority, extraction_status=extraction_status,
    )


relations = [
    R("r001", "project-atlas", "DEPENDS_ON", "service-vectordb-x", "src-atlas-arch-v2", "Project Atlas depends on VectorDB-X for relationship search."),
    R("r001-old", "project-atlas", "DEPENDS_ON", "service-legacy-graph", "src-atlas-arch-v1", "Project Atlas depended on LegacyGraph in 2024."),
    R("r002", "service-vectordb-x", "SUPPLIED_BY", "vendor-acme", "src-vectordb-vendor", "VectorDB-X is supplied by Acme Systems Inc.", "human_verified"),
    R("r002-marketing", "service-vectordb-x", "SUPPLIED_BY", "vendor-acme", "src-vendor-marketing", "VectorDB-X is supplied by Acme Systems Inc."),
    R("r003", "vendor-acme", "GOVERNED_BY", "reg-r17", "src-r17-register", "Acme Systems Inc. is governed by Regulation R-17.", "human_verified"),
    R("r004", "project-atlas", "OPERATED_BY", "owner-data", "src-atlas-arch-v2", "Project Atlas is operated by the Data Platform Team."),
    R("r005", "service-vectordb-x", "LOCATED_IN", "region-canada", "src-vectordb-controls", "is located in Canada."),
    R("r006", "service-vectordb-x", "IMPLEMENTS_CONTROL", "control-encryption", "src-vectordb-controls", "VectorDB-X implements the Encryption at Rest control"),
    R("r007", "control-encryption", "GOVERNED_BY", "reg-r17", "src-encryption-r17", "The Encryption at Rest control is governed by Regulation R-17."),
    R("r008", "project-atlas", "USES_PLATFORM", "platform-enterprise", "src-atlas-platform", "Project Atlas uses the Enterprise Platform."),
    R("r009", "platform-enterprise", "GOVERNED_BY", "reg-r17", "src-platform-r17", "The Enterprise Platform is governed by Regulation R-17."),
    R("r010", "service-atlas-search", "OPERATED_BY", "owner-search", "src-atlas-search", "Atlas Search Service is operated by the Search Reliability Team"),
    R("r011", "service-atlas-search", "IMPLEMENTS_CONTROL", "control-access", "src-atlas-search", "implements the Access Logging control"),
    R("r012", "control-access", "GOVERNED_BY", "reg-r22", "src-access-r22", "The Access Logging control is governed by Regulation R-22."),
    R("r013", "project-borealis", "DEPENDS_ON", "service-analytics", "src-borealis", "Project Borealis depends on Analytics Service."),
    R("r014", "service-analytics", "SUPPLIED_BY", "vendor-northwind", "src-analytics-vendor", "Analytics Service is supplied by Northwind Data."),
    R("r015", "vendor-northwind", "GOVERNED_BY", "reg-r22", "src-northwind-r22", "Northwind Data is governed by Regulation R-22."),
    R("r016", "project-checkout", "DEPENDS_ON", "service-payments", "src-payments", "Checkout Modernization depends on Payments API."),
    R("r017", "service-payments", "DEPENDS_ON", "database-orders", "src-payments", "Payments API depends on OrdersDB"),
    R("r018", "service-payments", "OPERATED_BY", "owner-commerce", "src-payments", "is operated by the Commerce Operations Team."),
    R("r019", "database-orders", "LOCATED_IN", "region-eu", "src-orders-region", "OrdersDB is located in the European Union."),
    R("r020", "service-risk", "DEPENDS_ON", "service-identity", "src-risk", "Risk Engine depends on Identity Service"),
    R("r021", "service-risk", "OPERATED_BY", "owner-risk", "src-risk", "is operated by the Risk Engineering Team."),
    R("r022", "service-identity", "SUPPLIED_BY", "vendor-contoso", "src-identity-vendor", "Identity Service is supplied by Contoso Identity."),
    R("r023", "vendor-contoso", "GOVERNED_BY", "reg-r17", "src-contoso-r17", "Contoso Identity is governed by Regulation R-17."),
    R("r024", "bu-platform", "OWNS", "service-shared-auth", "src-shared-auth-ownership", "Platform Engineering owns Shared Auth Service."),
    R("r025", "bu-risk", "OWNS", "service-shared-auth", "src-shared-auth-ownership", "Risk Business Unit owns Shared Auth Service."),
    R("r026", "bu-data", "OWNS", "project-atlas", "src-business-ownership", "Data Business Unit owns Project Atlas."),
    R("r027", "bu-commerce", "OWNS", "service-payments", "src-business-ownership", "Commerce Business Unit owns Payments API."),
    R("r028", "platform-enterprise", "HOSTS", "service-vectordb-x", "src-platform-hosting-a", "Enterprise Platform hosts VectorDB-X"),
    R("r029", "platform-enterprise", "HOSTS", "service-analytics", "src-platform-hosting-a", "Analytics Service"),
    R("r030", "platform-enterprise", "HOSTS", "service-payments", "src-platform-hosting-a", "Payments API."),
    R("r031", "platform-enterprise", "HOSTS", "service-risk", "src-platform-hosting-b", "Enterprise Platform hosts Risk Engine"),
    R("r032", "platform-enterprise", "HOSTS", "service-identity", "src-platform-hosting-b", "Identity Service"),
    R("r033", "platform-enterprise", "HOSTS", "service-atlas-search", "src-platform-hosting-b", "Atlas Search Service"),
    R("r034", "platform-enterprise", "HOSTS", "service-shared-auth", "src-platform-hosting-b", "Shared Auth Service."),
    R("g001", "globex-project-atlas", "DEPENDS_ON", "globex-vector-store", "src-globex-atlas", "Globex Project Atlas depends on Globex Vector Store."),
    R("g002", "globex-vector-store", "SUPPLIED_BY", "globex-vendor-acme", "src-globex-vendor", "Globex Vector Store is supplied by Acme Systems."),
    R("g003", "globex-vendor-acme", "GOVERNED_BY", "reg-r22", "src-globex-r22", "Acme Systems is governed by Regulation R-22 for Globex workloads."),
]

print("entities:", len(entities), "frozen relations:", len(relations))
assert len(entity_by_id) == len(entities)


### Optional live structured extraction

The live path returns names, types, controlled predicates, and verbatim spans. It never returns authoritative graph IDs. `resolve_entities()` maps proposed mentions to the registry; unresolved or ambiguous proposals are quarantined.


In [ ]:
class ExtractedEntityCandidate(BaseModel):
    mention: str
    entity_type: EntityType


class ExtractedRelationCandidate(BaseModel):
    source_mention: str
    source_type: EntityType
    relation_type: RelationType
    target_mention: str
    target_type: EntityType
    source_span: str


class ExtractedSourceGraph(BaseModel):
    entities: list[ExtractedEntityCandidate]
    relations: list[ExtractedRelationCandidate]


def live_extract_source(source: SourceRecord) -> ExtractedSourceGraph:
    if not (
        os.getenv("GRAPHRAG_USE_LIVE_EXTRACTION") == "1"
        and os.getenv("OPENAI_API_KEY")
        and os.getenv("GRAPHRAG_MODEL")
    ):
        raise RuntimeError("Live extraction is opt-in; frozen reviewed fixtures are the default.")
    from langchain_openai import ChatOpenAI

    model = ChatOpenAI(model=os.environ["GRAPHRAG_MODEL"], temperature=0).with_structured_output(
        ExtractedSourceGraph, method="json_schema"
    )
    return model.invoke(
        "Extract only explicit entities and allowed relationships. Copy each supporting span verbatim. "
        "Do not invent IDs or facts.\nAllowed relations: "
        f"{sorted(RELATION_SCHEMA)}\nSource ID: {source.source_id}\nText: {source.text}"
    )


extraction_mode = "live structured extraction available" if os.getenv("GRAPHRAG_USE_LIVE_EXTRACTION") == "1" else "frozen teaching extraction fixture"
print("mode:", extraction_mode)


## 6. Deterministically validate extraction records

Structured output is not trusted output. We validate source existence, endpoint IDs, controlled relation type, tenant scope, exact source-span containment, source version, status, and directional type schema. Invalid relations are quarantined rather than coerced.


In [ ]:
def validate_relation(record: RelationRecord) -> list[str]:
    errors: list[str] = []
    source = source_by_id.get(record.source_id)
    source_entity = entity_by_id.get(record.source_entity_id)
    target_entity = entity_by_id.get(record.target_entity_id)
    if source is None:
        return ["unknown_source_id"]
    if source_entity is None:
        errors.append("unknown_source_entity")
    if target_entity is None:
        errors.append("unknown_target_entity")
    if record.relation_type not in RELATION_SCHEMA:
        errors.append("unknown_relation_type")
    if record.tenant_id != source.tenant_id:
        errors.append("tenant_source_mismatch")
    if record.source_span not in source.text:
        errors.append("source_span_not_found")
    if record.source_version != source.version:
        errors.append("source_version_mismatch")
    if record.status != source.status:
        errors.append("source_status_mismatch")
    if source_entity and target_entity:
        allowed_sources, allowed_targets = RELATION_SCHEMA.get(record.relation_type, (set(), set()))
        if source_entity.entity_type not in allowed_sources or target_entity.entity_type not in allowed_targets:
            errors.append("direction_or_type_schema_violation")
        for endpoint in (source_entity, target_entity):
            if endpoint.tenant_id not in {record.tenant_id, "public"}:
                errors.append("endpoint_tenant_mismatch")
                break
    return errors


validated_relations, quarantined = [], []
for relation in relations:
    errors = validate_relation(relation)
    (quarantined if errors else validated_relations).append((relation, errors) if errors else relation)

print("validated:", len(validated_relations), "quarantined:", len(quarantined))
assert len(validated_relations) == len(relations)

# Failure injection: wrong direction, missing source, invented span, and tenant mismatch.
malformed = [
    relations[0].model_copy(update={"relation_id": "bad-direction", "source_entity_id": "reg-r17", "target_entity_id": "project-atlas"}),
    relations[0].model_copy(update={"relation_id": "bad-source", "source_id": "missing-source"}),
    relations[0].model_copy(update={"relation_id": "bad-span", "source_span": "Invented relationship."}),
    relations[0].model_copy(update={"relation_id": "bad-tenant", "tenant_id": "globex"}),
]
malformed_results = pd.DataFrame([
    {"relation_id": relation.relation_id, "quarantine_reasons": validate_relation(relation)}
    for relation in malformed
])
display(malformed_results)
assert malformed_results.quarantine_reasons.map(bool).all()


## 7. Entity resolution is an evaluated subsystem

The teaching resolver uses normalized aliases, type hints, tenant scope, and the authoritative registry. It deliberately does not claim to solve universal entity resolution. We compare it with two flawed policies:

- **false merge:** collapse entities by a broad alias while ignoring type and tenant;
- **false split:** use only exact canonical strings and miss aliases such as `Acme Systems Inc.`.


In [ ]:
def normalize_name(value: str) -> str:
    value = re.sub(r"[^a-z0-9]+", " ", value.lower()).strip()
    return re.sub(r"\b(inc|incorporated|corp|corporation)\b", "", value).strip()


def resolve_mention(mention: str, principal: Principal, type_hint: str | None = None) -> QueryEntity:
    normalized = normalize_name(mention)
    candidates = []
    for entity in entities:
        if entity.tenant_id not in {principal.tenant_id, "public"}:
            continue
        if type_hint and entity.entity_type != type_hint:
            continue
        names = [entity.canonical_name, *entity.aliases]
        if normalized in {normalize_name(name) for name in names}:
            candidates.append(entity.entity_id)
    if len(candidates) == 1:
        return QueryEntity(mention=mention, resolved_entity_id=candidates[0], candidate_entity_ids=candidates, status="resolved")
    if len(candidates) > 1:
        return QueryEntity(mention=mention, candidate_entity_ids=sorted(candidates), status="ambiguous")
    return QueryEntity(mention=mention, candidate_entity_ids=[], status="unresolved")


resolution_gold = [
    ("Acme Systems", "Vendor", "acme", "vendor-acme"),
    ("Acme Systems Inc.", "Vendor", "acme", "vendor-acme"),
    ("ACME", "Vendor", "acme", "vendor-acme"),
    ("Project Atlas", "Project", "acme", "project-atlas"),
    ("Atlas Search", "Service", "acme", "service-atlas-search"),
    ("Acme Systems", "Vendor", "globex", "globex-vendor-acme"),
]
resolution_rows = []
for mention, entity_type, tenant, expected in resolution_gold:
    result = resolve_mention(mention, Principal(f"user-{tenant}", tenant), entity_type)
    resolution_rows.append({"mention": mention, "type": entity_type, "tenant": tenant, "expected": expected, "predicted": result.resolved_entity_id, "correct": result.resolved_entity_id == expected})
resolution_df = pd.DataFrame(resolution_rows)
display(resolution_df)
assert resolution_df.correct.all()

# Controlled false merge: an over-broad "Atlas" key merges a project and service.
bad_merge_clusters = {"project-atlas": "atlas", "service-atlas-search": "atlas"}
false_merges = int(bad_merge_clusters["project-atlas"] == bad_merge_clusters["service-atlas-search"])

# Controlled false split: exact-string identity separates aliases of one vendor.
bad_split_clusters = {"Acme Systems": "acme systems", "Acme Systems Inc.": "acme systems inc"}
false_splits = int(bad_split_clusters["Acme Systems"] != bad_split_clusters["Acme Systems Inc."])

resolution_metrics = {
    "correct_resolutions": int(resolution_df.correct.sum()),
    "false_merge_demo": false_merges,
    "false_split_demo": false_splits,
}
display(pd.Series(resolution_metrics, name="count").to_frame())
assert false_merges == 1 and false_splits == 1


## 8. Build a provenance-preserving `MultiDiGraph`

`MultiDiGraph` is appropriate because the same entity pair can have multiple relation types or multiple source-backed assertions. A `DiGraph` would overwrite edge attributes for repeated `(u, v)` pairs. The trade-off is more complex edge access: every path step must retain its edge key (`relation_id`).


In [ ]:
def build_graph(entity_records: list[EntityRecord], relation_records: list[RelationRecord]) -> nx.MultiDiGraph:
    graph = nx.MultiDiGraph()
    for entity in entity_records:
        graph.add_node(entity.entity_id, **entity.model_dump())
    for relation in relation_records:
        if validate_relation(relation):
            continue
        graph.add_edge(
            relation.source_entity_id,
            relation.target_entity_id,
            key=relation.relation_id,
            **relation.model_dump(),
        )
    return graph


G = build_graph(entities, validated_relations)
print("graph nodes:", G.number_of_nodes(), "graph edges:", G.number_of_edges())
parallel = G.get_edge_data("service-vectordb-x", "vendor-acme")
display(pd.DataFrame(parallel.values())[["relation_id", "source_id", "authority", "source_version"]])
assert set(parallel) == {"r002", "r002-marketing"}
assert all(data.get("source_id") and data.get("source_span") for *_, data in G.edges(keys=True, data=True))

# Visualize the current Acme subgraph; color encodes entity type, not importance.
current_acme_edges = [(u, v, k) for u, v, k, d in G.edges(keys=True, data=True) if d["tenant_id"] == "acme" and d["status"] == "current"]
H = G.edge_subgraph(current_acme_edges).copy()
type_color = {"Project": "#2F6BFF", "Service": "#16A3A5", "Vendor": "#F59E42", "Regulation": "#EF4444", "Control": "#8B5CF6", "Platform": "#64748B"}
plt.figure(figsize=(13, 9))
pos = nx.spring_layout(H, seed=7, k=0.9)
nx.draw_networkx_nodes(H, pos, node_size=650, node_color=[type_color.get(H.nodes[n]["entity_type"], "#CBD5E1") for n in H], alpha=0.9)
nx.draw_networkx_edges(H, pos, arrows=True, arrowstyle="-|>", width=1.0, alpha=0.45, connectionstyle="arc3,rad=0.05")
nx.draw_networkx_labels(H, pos, labels={n: H.nodes[n]["canonical_name"] for n in H}, font_size=7)
plt.title("Validated current Acme property graph")
plt.axis("off")
plt.show()


## 9. Preserve the original manual Atlas trace—now as structured source-backed evidence

Before building a general retriever, inspect the three facts manually. This retains the original lesson: a multi-hop answer is only as trustworthy as every edge and its provenance.


In [ ]:
manual_relation_ids = ["r001", "r002", "r003"]
manual_facts = [next(relation for relation in validated_relations if relation.relation_id == relation_id) for relation_id in manual_relation_ids]
manual_table = pd.DataFrame([{
    "fact": relation.relation_id,
    "subject": entity_by_id[relation.source_entity_id].canonical_name,
    "predicate": relation.relation_type,
    "object": entity_by_id[relation.target_entity_id].canonical_name,
    "source": relation.source_id,
    "version": relation.source_version,
    "span": relation.source_span,
} for relation in manual_facts])
display(manual_table)
assert manual_table.source.notna().all()


## 10. Failure injection: undirected connectivity fabricates semantics

The old `G.to_undirected()` approach is retained only as a failure demonstration. If two business units both `OWNS → Shared Auth Service`, an undirected graph invents a path between the owners. Direction-aware traversal correctly blocks it.


In [ ]:
undirected_shortcut = nx.shortest_path(G.to_undirected(), "bu-platform", "bu-risk")
print("misleading undirected path:", " → ".join(undirected_shortcut))
assert undirected_shortcut == ["bu-platform", "service-shared-auth", "bu-risk"]

def directed_relation_path_exists(graph: nx.MultiDiGraph, start: str, target: str, allowed_relations: set[str]) -> bool:
    queue = deque([start])
    seen = {start}
    while queue:
        node = queue.popleft()
        for _, neighbor, _, data in graph.out_edges(node, keys=True, data=True):
            if data["relation_type"] not in allowed_relations or data["status"] != "current":
                continue
            if neighbor == target:
                return True
            if neighbor not in seen:
                seen.add(neighbor)
                queue.append(neighbor)
    return False

assert not directed_relation_path_exists(G, "bu-platform", "bu-risk", {"OWNS"})
print("direction-aware OWNS traversal: blocked")


## 11. Bounded, tenant-aware, relation-aware weighted traversal

`shortest ≠ most meaningful`. Atlas has a two-hop path through the generic Enterprise Platform and a three-hop supplier/compliance path. Unweighted shortest path prefers the generic hub. The bounded retriever uses explicit relation policies, source authority, lifecycle status, tenant scope, and semantic edge costs.


In [ ]:
class TraversalPolicy(BaseModel):
    max_hops: int = MAX_HOPS
    max_facts: int = MAX_FACTS
    allowed_relations: set[str]
    reverse_relations: set[str] = Field(default_factory=set)
    allowed_statuses: set[str] = Field(default_factory=lambda: {"current"})
    allowed_authorities: set[str] = Field(default_factory=lambda: set(GENERATION_AUTHORITIES))


def node_visible(graph: nx.MultiDiGraph, node_id: str, principal: Principal) -> bool:
    return graph.nodes[node_id]["tenant_id"] in {principal.tenant_id, "public"}


def edge_eligible(graph: nx.MultiDiGraph, u: str, v: str, data: dict, principal: Principal, policy: TraversalPolicy) -> bool:
    return (
        data["tenant_id"] in {principal.tenant_id, "public"}
        and data["status"] in policy.allowed_statuses
        and data["authority"] in policy.allowed_authorities
        and data["relation_type"] in policy.allowed_relations
        and node_visible(graph, u, principal)
        and node_visible(graph, v, principal)
        and bool(data.get("source_id") and data.get("source_span") and data.get("source_version"))
    )


def candidate_steps(graph: nx.MultiDiGraph, node_id: str, principal: Principal, policy: TraversalPolicy):
    for u, v, key, data in graph.out_edges(node_id, keys=True, data=True):
        if edge_eligible(graph, u, v, data, principal, policy):
            yield v, key, data, "forward"
    for u, v, key, data in graph.in_edges(node_id, keys=True, data=True):
        if data["relation_type"] in policy.reverse_relations and edge_eligible(graph, u, v, data, principal, policy):
            yield u, key, data, "reverse"


def path_edge(data: dict, direction: str) -> PathEdge:
    return PathEdge(
        relation_id=data["relation_id"], source_entity_id=data["source_entity_id"],
        relation_type=data["relation_type"], target_entity_id=data["target_entity_id"],
        source_id=data["source_id"], source_span=data["source_span"],
        source_version=data["source_version"], traversal_direction=direction,
    )


def retrieve_path(graph: nx.MultiDiGraph, start: str, target: str, principal: Principal, policy: TraversalPolicy) -> GraphPath:
    counter = itertools.count()
    heap = [(0.0, 0, next(counter), start, [], frozenset({start}))]
    nodes_seen = {start}
    edges_evaluated = 0
    max_depth = 0
    while heap:
        cost, hops, _, node, edges, visited = heapq.heappop(heap)
        max_depth = max(max_depth, hops)
        if node == target:
            trace = TraversalTrace(nodes_visited=len(nodes_seen), edges_evaluated=edges_evaluated, facts_returned=len(edges), max_depth=max_depth)
            return GraphPath(seed_entity_ids=[start], target_entity_id=target, edges=edges, terminal_reason="path_found", trace=trace)
        if hops >= policy.max_hops:
            continue
        for neighbor, _, data, direction in candidate_steps(graph, node, principal, policy):
            edges_evaluated += 1
            if edges_evaluated > policy.max_facts:
                trace = TraversalTrace(nodes_visited=len(nodes_seen), edges_evaluated=edges_evaluated, facts_returned=0, max_depth=max_depth, stopped_by_budget=True)
                return GraphPath(seed_entity_ids=[start], target_entity_id=target, edges=[], terminal_reason="fact_budget_exhausted", trace=trace)
            if neighbor in visited:
                continue
            nodes_seen.add(neighbor)
            edge = path_edge(data, direction)
            authority_penalty = 0.0 if data["authority"] in {"official_record", "regulator"} else 0.5
            next_cost = cost + RELATION_COST[data["relation_type"]] + authority_penalty
            heapq.heappush(heap, (next_cost, hops + 1, next(counter), neighbor, [*edges, edge], visited | {neighbor}))
    trace = TraversalTrace(nodes_visited=len(nodes_seen), edges_evaluated=edges_evaluated, facts_returned=0, max_depth=max_depth)
    return GraphPath(seed_entity_ids=[start], target_entity_id=target, edges=[], terminal_reason="no_path_within_policy", trace=trace)


acme = Principal("user-123", "acme")
supplier_policy = TraversalPolicy(allowed_relations={"DEPENDS_ON", "SUPPLIED_BY", "GOVERNED_BY"})
atlas_path = retrieve_path(G, "project-atlas", "reg-r17", acme, supplier_policy)
print("selected relation path:", [edge.relation_id for edge in atlas_path.edges])
display(pd.DataFrame([edge.model_dump() for edge in atlas_path.edges]))
assert [edge.relation_id for edge in atlas_path.edges] == ["r001", "r002", "r003"]
assert atlas_path.trace.max_depth <= MAX_HOPS and atlas_path.trace.edges_evaluated <= MAX_FACTS

naive_shortest = nx.shortest_path(G, "project-atlas", "reg-r17")
print("unweighted shortest nodes:", naive_shortest)
assert naive_shortest == ["project-atlas", "platform-enterprise", "reg-r17"]


## 12. High-degree hub experiment

Naive one-hop expansion around `Enterprise Platform` pulls seven hosted services plus other edges. A bounded relation policy stops fan-out after three evaluated facts. The point is not that three is universally correct; it is that expansion must have an explicit budget.


In [ ]:
def naive_neighbor_edges(graph: nx.MultiDiGraph, node_id: str) -> list[tuple]:
    return list(graph.out_edges(node_id, keys=True, data=True)) + list(graph.in_edges(node_id, keys=True, data=True))


def bounded_expand(graph: nx.MultiDiGraph, start: str, principal: Principal, policy: TraversalPolicy) -> tuple[list[PathEdge], TraversalTrace]:
    queue = deque([(start, 0)])
    seen = {start}
    facts: list[PathEdge] = []
    evaluated = 0
    max_depth = 0
    while queue and len(facts) < policy.max_facts:
        node, depth = queue.popleft()
        max_depth = max(max_depth, depth)
        if depth >= policy.max_hops:
            continue
        for neighbor, _, data, direction in candidate_steps(graph, node, principal, policy):
            evaluated += 1
            if len(facts) >= policy.max_facts:
                break
            facts.append(path_edge(data, direction))
            if neighbor not in seen:
                seen.add(neighbor)
                queue.append((neighbor, depth + 1))
    return facts, TraversalTrace(nodes_visited=len(seen), edges_evaluated=evaluated, facts_returned=len(facts), max_depth=max_depth, stopped_by_budget=bool(queue or len(facts) >= policy.max_facts))


naive_hub = naive_neighbor_edges(G, "platform-enterprise")
bounded_hub, bounded_trace = bounded_expand(
    G, "platform-enterprise", acme,
    TraversalPolicy(max_hops=1, max_facts=3, allowed_relations={"HOSTS"}),
)
hub_comparison = pd.DataFrame([
    {"method": "naive incident edges", "facts": len(naive_hub), "bounded": False},
    {"method": "policy-bounded HOSTS", "facts": len(bounded_hub), "bounded": bounded_trace.stopped_by_budget},
])
display(hub_comparison)
assert len(bounded_hub) == 3 and len(naive_hub) > len(bounded_hub)


## 13. Query entity resolution and ambiguity

The model or fixture proposes mentions—not graph IDs. The application resolves each mention against visible, typed entities. `Atlas` is ambiguous between a project and a service and therefore requires clarification. We never choose arbitrarily.


In [ ]:
class QueryMentions(BaseModel):
    mentions: list[str]


def resolve_query_entities(query: str, mentions: list[str], principal: Principal, type_hints: dict[str, str] | None = None) -> list[QueryEntity]:
    type_hints = type_hints or {}
    return [resolve_mention(mention, principal, type_hints.get(mention)) for mention in mentions]


central_resolution = resolve_query_entities(
    "Does the supplier behind Project Atlas comply with Regulation R-17?",
    ["Project Atlas", "Regulation R-17"], acme,
    {"Project Atlas": "Project", "Regulation R-17": "Regulation"},
)
ambiguous_resolution = resolve_query_entities("How is Atlas related to R-17?", ["Atlas", "R-17"], acme)
display(pd.DataFrame([item.model_dump() for item in central_resolution + ambiguous_resolution]))
assert all(item.status == "resolved" for item in central_resolution)
assert ambiguous_resolution[0].status == "ambiguous"


## 14. Hybrid text-to-graph seeding

Exact entity names are not always present in the question. The small hybrid pattern retrieves source text with BM25, maps those sources to validated entity IDs, then uses those IDs as bounded graph seeds. This is deliberately lightweight—the lesson remains graph control, not another dense-retrieval course.

![Text retrieval can seed bounded graph expansion, which returns source spans for answering.](assets/hybrid-graph-text.svg)


In [ ]:
source_tokens = [re.findall(r"[a-z0-9-]+", source.text.lower()) for source in sources]
source_bm25 = BM25Okapi(source_tokens)
relations_by_source = defaultdict(list)
for relation in validated_relations:
    relations_by_source[relation.source_id].append(relation)


def text_search(query: str, principal: Principal, k: int = 2) -> list[SourceRecord]:
    scores = source_bm25.get_scores(re.findall(r"[a-z0-9-]+", query.lower()))
    ranked = sorted(zip(sources, scores), key=lambda pair: pair[1], reverse=True)
    return [source for source, score in ranked if score > 0 and source.tenant_id in {principal.tenant_id, "public"}][:k]


def hybrid_text_to_graph_seeds(query: str, principal: Principal, k: int = 2) -> tuple[list[SourceRecord], list[str]]:
    source_hits = text_search(query, principal, k=k)
    seed_ids: list[str] = []
    for source in source_hits:
        for relation in relations_by_source[source.source_id]:
            for entity_id in (relation.source_entity_id, relation.target_entity_id):
                if entity_by_id[entity_id].tenant_id in {principal.tenant_id, "public"} and entity_id not in seed_ids:
                    seed_ids.append(entity_id)
    return source_hits, seed_ids


paraphrased_query = "Which approved supplier stands behind the relationship datastore used by the flagship project?"
hybrid_sources, hybrid_seeds = hybrid_text_to_graph_seeds(paraphrased_query, acme, k=3)
display(pd.DataFrame({"source_hit": [s.source_id for s in hybrid_sources]}))
print("linked entity seeds:", hybrid_seeds)
assert "vendor-acme" in hybrid_seeds or "service-vectordb-x" in hybrid_seeds


## 15. Hydrate path facts into request-local evidence and generate only from validated sources

Path generation and answer generation are separate. The retriever returns structured facts first. Hydration assigns request-local evidence IDs and carries the exact source span, document, and version. The deterministic renderer is the default; optional live synthesis receives only this validated evidence.


In [ ]:
def hydrate_path_sources(path: GraphPath) -> list[EvidenceRecord]:
    evidence: list[EvidenceRecord] = []
    for index, edge in enumerate(path.edges, start=1):
        source = source_by_id[edge.source_id]
        assert edge.source_span in source.text
        source_name = entity_by_id[edge.source_entity_id].canonical_name
        target_name = entity_by_id[edge.target_entity_id].canonical_name
        evidence.append(EvidenceRecord(
            evidence_id=f"E{index}", relation_id=edge.relation_id, source_id=edge.source_id,
            document_id=source.document_id, source_version=edge.source_version,
            source_span=edge.source_span,
            statement=f"{source_name} {edge.relation_type} {target_name}",
        ))
    return evidence


def render_grounded_answer(question: str, evidence: list[EvidenceRecord]) -> GroundedAnswer:
    if os.getenv("GRAPHRAG_USE_LIVE_GENERATOR") == "1" and os.getenv("OPENAI_API_KEY") and os.getenv("GRAPHRAG_MODEL"):
        from langchain_openai import ChatOpenAI
        model = ChatOpenAI(model=os.environ["GRAPHRAG_MODEL"], temperature=0).with_structured_output(GroundedAnswer, method="json_schema")
        return model.invoke(
            "Answer only from the supplied source-backed graph evidence. Cite every claim with an evidence ID. "
            f"Question: {question}\nEvidence: {[item.model_dump() for item in evidence]}"
        )
    text = " ".join(f"{item.statement} [{item.evidence_id}]." for item in evidence)
    return GroundedAnswer(text=text, citation_ids=[item.evidence_id for item in evidence])


atlas_evidence = hydrate_path_sources(atlas_path)
atlas_answer = render_grounded_answer("How is Project Atlas connected to Regulation R-17?", atlas_evidence)
display(pd.DataFrame([item.model_dump() for item in atlas_evidence]))
print(atlas_answer.text)
known_ids = {item.evidence_id for item in atlas_evidence}
assert set(atlas_answer.citation_ids).issubset(known_ids)
assert all(item.source_span and item.source_version for item in atlas_evidence)


## 16. False merge and false split change reachability

These are not abstract data-cleaning errors:

- merging `Project Atlas` with `Atlas Search Service` fabricates a path from the project to R-22;
- splitting `Acme Systems Inc.` from `Acme Systems` breaks the valid Atlas → R-17 chain.


In [ ]:
false_merge_graph = nx.relabel_nodes(G, {"service-atlas-search": "project-atlas"}, copy=True)
fabricated = nx.has_path(false_merge_graph, "project-atlas", "reg-r22")

false_split_graph = G.copy()
false_split_graph.add_node("vendor-acme-split", canonical_name="Acme Systems Inc.", entity_type="Vendor", aliases=[], tenant_id="acme")
edge_data = false_split_graph.get_edge_data("service-vectordb-x", "vendor-acme", "r002")
false_split_graph.remove_edges_from([
    ("service-vectordb-x", "vendor-acme", key)
    for key in list(false_split_graph["service-vectordb-x"]["vendor-acme"])
])
false_split_graph.add_edge("service-vectordb-x", "vendor-acme-split", key="r002", **edge_data)
split_result = retrieve_path(false_split_graph, "project-atlas", "reg-r17", acme, supplier_policy)
split_path_exists = split_result.terminal_reason == "path_found"

display(pd.DataFrame([
    {"failure": "false merge", "observed": fabricated, "meaning": "fabricated Project Atlas → R-22 path"},
    {"failure": "false split", "observed": not split_path_exists, "meaning": "valid supplier chain becomes unreachable"},
]))
assert fabricated and not split_path_exists


## 17. Freshness, source replacement, and stale-edge reconciliation

Graph topology is versioned derived data. Current retrieval excludes the historical `Project Atlas DEPENDS_ON LegacyGraph` relation. Updating a source must remove its previous derived edges before adding replacement facts.


In [ ]:
current_policy = TraversalPolicy(allowed_relations={"DEPENDS_ON"})
current_to_legacy = retrieve_path(G, "project-atlas", "service-legacy-graph", acme, current_policy)
historical_policy = TraversalPolicy(allowed_relations={"DEPENDS_ON"}, allowed_statuses={"historical"})
historical_to_legacy = retrieve_path(G, "project-atlas", "service-legacy-graph", acme, historical_policy)
assert current_to_legacy.terminal_reason != "path_found"
assert historical_to_legacy.terminal_reason == "path_found"


def replace_source_relations(graph: nx.MultiDiGraph, source_id: str, replacements: list[RelationRecord]) -> nx.MultiDiGraph:
    updated = graph.copy()
    to_remove = [(u, v, key) for u, v, key, data in updated.edges(keys=True, data=True) if data["source_id"] == source_id]
    updated.remove_edges_from(to_remove)
    for relation in replacements:
        updated.add_edge(relation.source_entity_id, relation.target_entity_id, key=relation.relation_id, **relation.model_dump())
    return updated


source_v3 = S("src-atlas-arch-v3", "atlas-architecture", "Project Atlas depends on VectorDB-Y for relationship search.", version="3")
source_by_id[source_v3.source_id] = source_v3
replacement = R("r001-v3", "project-atlas", "DEPENDS_ON", "service-vectordb-y", source_v3.source_id, source_v3.text, "human_verified")
G_v3 = replace_source_relations(G, "src-atlas-arch-v2", [replacement])
remaining_old = [data for *_, data in G_v3.edges(keys=True, data=True) if data["source_id"] == "src-atlas-arch-v2"]
assert not remaining_old and G_v3.has_edge("project-atlas", "service-vectordb-y", "r001-v3")
print("source-level replacement removed stale derived edges and added r001-v3")


## 18. Tenant-aware traversal and structural leakage

Authorization is enforced while expanding candidates, not after a path is built. To make structural leakage concrete, we inject an invalid Globex-scoped shortcut between Acme nodes. Generic NetworkX finds it; the authorized retriever never evaluates it as eligible evidence.


In [ ]:
leaky_graph = G.copy()
leaky_graph.add_edge(
    "project-borealis", "reg-r17", key="leak-001",
    relation_id="leak-001", source_entity_id="project-borealis", relation_type="DEPENDS_ON",
    target_entity_id="reg-r17", source_id="src-globex-atlas", source_span="Globex Project Atlas depends on Globex Vector Store.",
    source_version="1", tenant_id="globex", status="current", authority="official_record", extraction_status="frozen_fixture",
)
assert nx.has_path(leaky_graph, "project-borealis", "reg-r17")
safe_leak_test = retrieve_path(
    leaky_graph, "project-borealis", "reg-r17", acme,
    TraversalPolicy(max_hops=1, allowed_relations={"DEPENDS_ON"}),
)
assert safe_leak_test.terminal_reason != "path_found"


def path_scope_violations(path: GraphPath, principal: Principal) -> tuple[int, int]:
    unauthorized_edges = sum(source_by_id[edge.source_id].tenant_id not in {principal.tenant_id, "public"} for edge in path.edges)
    node_ids = {node for edge in path.edges for node in (edge.source_entity_id, edge.target_entity_id)}
    unauthorized_nodes = sum(entity_by_id[node].tenant_id not in {principal.tenant_id, "public"} for node in node_ids)
    return unauthorized_nodes, unauthorized_edges


assert path_scope_violations(atlas_path, acme) == (0, 0)
print("unfiltered graph: shortcut exists; authorized traversal: shortcut blocked")


## 19. Extraction quality is evaluated before answer quality

We compare a small predicted extraction fixture with gold triples. Wrong direction, wrong type, missing, and invented relations affect precision, recall, and direction accuracy even before retrieval begins.


In [ ]:
gold_triples = {
    ("project-atlas", "DEPENDS_ON", "service-vectordb-x"),
    ("service-vectordb-x", "SUPPLIED_BY", "vendor-acme"),
    ("vendor-acme", "GOVERNED_BY", "reg-r17"),
    ("service-vectordb-x", "IMPLEMENTS_CONTROL", "control-encryption"),
    ("control-encryption", "GOVERNED_BY", "reg-r17"),
}
predicted_triples = {
    ("project-atlas", "DEPENDS_ON", "service-vectordb-x"),
    ("service-vectordb-x", "SUPPLIED_BY", "vendor-acme"),
    ("reg-r17", "GOVERNED_BY", "vendor-acme"),          # wrong direction
    ("service-vectordb-x", "IMPLEMENTS_CONTROL", "control-encryption"),
    ("project-atlas", "GOVERNED_BY", "reg-r17"),       # invented relation
}
true_positive = len(gold_triples & predicted_triples)
relation_precision = true_positive / len(predicted_triples)
relation_recall = true_positive / len(gold_triples)

gold_unordered = {(frozenset((s, t)), r) for s, r, t in gold_triples}
direction_candidates = [(s, r, t) for s, r, t in predicted_triples if (frozenset((s, t)), r) in gold_unordered]
direction_correct = sum((s, r, t) in gold_triples for s, r, t in direction_candidates)
direction_accuracy = direction_correct / len(direction_candidates)

extraction_metrics = pd.Series({
    "relation_precision": relation_precision,
    "relation_recall": relation_recall,
    "direction_accuracy": direction_accuracy,
}, name="measured")
display(extraction_metrics.to_frame())


## 20. Labelled path-retrieval dataset

The same 22 cases evaluate direct/one-hop, two-hop, three-hop, reverse traversal, aliases, ambiguity, a generic-hub trap, no path, cross-tenant resolution, and historical topology. Each successful case names expected relation IDs; negative cases name an expected terminal reason.


In [ ]:
class PathEvalCase(BaseModel):
    case_id: str
    slice: str
    query: str
    start_mention: str
    target_mention: str
    start_type: str | None = None
    target_type: str | None = None
    tenant_id: str = "acme"
    allowed_relations: set[str]
    reverse_relations: set[str] = Field(default_factory=set)
    allowed_statuses: set[str] = Field(default_factory=lambda: {"current"})
    expected_relation_ids: list[str] = Field(default_factory=list)
    expected_terminal: str = "path_found"


def Q(case_id, slice_, query, start, target, allowed, expected=(), **kwargs):
    return PathEvalCase(
        case_id=case_id, slice=slice_, query=query, start_mention=start, target_mention=target,
        allowed_relations=set(allowed), expected_relation_ids=list(expected), **kwargs,
    )


evaluation_cases = [
    Q("D1", "direct_lookup", "Who operates Project Atlas?", "Project Atlas", "Data Platform Team", {"OPERATED_BY"}, ["r004"], start_type="Project", target_type="Owner"),
    Q("D2", "direct_lookup", "Where is VectorDB-X located?", "VectorDB-X", "Canada", {"LOCATED_IN"}, ["r005"]),
    Q("D3", "direct_lookup", "Which control does VectorDB-X implement?", "VectorDB-X", "Encryption at Rest", {"IMPLEMENTS_CONTROL"}, ["r006"]),
    Q("D4", "direct_lookup", "What service does Borealis depend on?", "Project Borealis", "Analytics Service", {"DEPENDS_ON"}, ["r013"]),
    Q("H1", "two_hop", "Which supplier supports Project Atlas?", "Project Atlas", "Acme Systems", {"DEPENDS_ON", "SUPPLIED_BY"}, ["r001", "r002"], start_type="Project", target_type="Vendor"),
    Q("H2", "two_hop", "How is VectorDB-X connected to R-17?", "VectorDB-X", "R-17", {"SUPPLIED_BY", "GOVERNED_BY"}, ["r002", "r003"]),
    Q("H3", "two_hop", "How is Atlas Search Service governed by R-22?", "Atlas Search Service", "R-22", {"IMPLEMENTS_CONTROL", "GOVERNED_BY"}, ["r011", "r012"], start_type="Service"),
    Q("H4", "two_hop", "Which vendor backs Borealis?", "Project Borealis", "Northwind Data", {"DEPENDS_ON", "SUPPLIED_BY"}, ["r013", "r014"]),
    Q("M1", "three_hop", "Does the supplier of Atlas fall under R-17?", "Project Atlas", "R-17", {"DEPENDS_ON", "SUPPLIED_BY", "GOVERNED_BY"}, ["r001", "r002", "r003"], start_type="Project"),
    Q("M2", "three_hop", "How is Borealis connected to R-22?", "Project Borealis", "R-22", {"DEPENDS_ON", "SUPPLIED_BY", "GOVERNED_BY"}, ["r013", "r014", "r015"]),
    Q("M3", "three_hop", "Which regulation reaches Risk Engine through its identity vendor?", "Risk Engine", "R-17", {"DEPENDS_ON", "SUPPLIED_BY", "GOVERNED_BY"}, ["r020", "r022", "r023"]),
    Q("M4", "three_hop", "Where is Checkout Modernization's database located?", "Checkout Modernization", "European Union", {"DEPENDS_ON", "LOCATED_IN"}, ["r016", "r017", "r019"]),
    Q("R1", "direction_sensitive", "Which service is supplied by ACME?", "ACME", "VectorDB-X", {"SUPPLIED_BY"}, ["r002"], reverse_relations={"SUPPLIED_BY"}, start_type="Vendor", target_type="Service"),
    Q("R2", "direction_sensitive", "Which vendor is governed by R-17?", "R-17", "Acme Systems", {"GOVERNED_BY"}, ["r003"], reverse_relations={"GOVERNED_BY"}, start_type="Regulation", target_type="Vendor"),
    Q("A1", "alias_resolution", "How is ACME connected to R-17?", "ACME", "R-17", {"GOVERNED_BY"}, ["r003"], start_type="Vendor"),
    Q("A2", "alias_resolution", "How is Atlas Search tied to R-22?", "Atlas Search", "R-22", {"IMPLEMENTS_CONTROL", "GOVERNED_BY"}, ["r011", "r012"], start_type="Service"),
    Q("A3", "ambiguous_entity", "How is Atlas related to R-17?", "Atlas", "R-17", {"DEPENDS_ON", "SUPPLIED_BY", "GOVERNED_BY"}, expected_terminal="clarification_required"),
    Q("T1", "hub_trap", "Use the supplier chain from Project Atlas to R-17.", "Project Atlas", "R-17", set(RELATION_SCHEMA), ["r001", "r002", "r003"], start_type="Project"),
    Q("N1", "no_path", "How is Payments API connected to R-22?", "Payments API", "R-22", {"DEPENDS_ON", "SUPPLIED_BY", "GOVERNED_BY"}, expected_terminal="no_path_within_policy"),
    Q("X1", "cross_tenant", "How is Globex Project Atlas connected to R-22?", "Globex Project Atlas", "R-22", {"DEPENDS_ON", "SUPPLIED_BY", "GOVERNED_BY"}, expected_terminal="unresolved_entity"),
    Q("V1", "historical", "What did Project Atlas depend on in 2024?", "Project Atlas", "LegacyGraph", {"DEPENDS_ON"}, ["r001-old"], allowed_statuses={"historical"}, start_type="Project"),
    Q("O1", "ownership", "Which business unit owns Project Atlas?", "Project Atlas", "Data Business Unit", {"OWNS"}, ["r026"], reverse_relations={"OWNS"}, start_type="Project", target_type="BusinessUnit"),
]
assert 20 <= len(evaluation_cases) <= 30
print("evaluation cases:", len(evaluation_cases))


## 21. Run graph and text baselines on the same cases

The graph baseline evaluates expected relation IDs. The text baseline retrieves two source records and evaluates expected source coverage. This is not designed to force a winner: direct lookups often work well with text, while multi-hop relationship questions are where explicit topology should add value.


In [ ]:
relation_by_id = {relation.relation_id: relation for relation in validated_relations}


def run_graph_case(case: PathEvalCase) -> dict:
    principal = Principal(f"eval-{case.case_id}", case.tenant_id)
    start = resolve_mention(case.start_mention, principal, case.start_type)
    target = resolve_mention(case.target_mention, principal, case.target_type)
    if start.status == "ambiguous" or target.status == "ambiguous":
        terminal = "clarification_required"
        path = GraphPath(seed_entity_ids=[], target_entity_id=None, terminal_reason=terminal, trace=TraversalTrace())
    elif start.status != "resolved" or target.status != "resolved":
        terminal = "unresolved_entity"
        path = GraphPath(seed_entity_ids=[], target_entity_id=None, terminal_reason=terminal, trace=TraversalTrace())
    else:
        policy = TraversalPolicy(
            allowed_relations=case.allowed_relations,
            reverse_relations=case.reverse_relations,
            allowed_statuses=case.allowed_statuses,
        )
        path = retrieve_path(G, start.resolved_entity_id, target.resolved_entity_id, principal, policy)
        terminal = path.terminal_reason
    predicted = [edge.relation_id for edge in path.edges]
    expected = case.expected_relation_ids
    intersection = len(set(predicted) & set(expected))
    precision = intersection / len(predicted) if predicted else (1.0 if not expected else 0.0)
    recall = intersection / len(expected) if expected else (1.0 if not predicted else 0.0)
    provenance = (
        sum(bool(edge.source_id and edge.source_span and edge.source_version) for edge in path.edges) / len(path.edges)
        if path.edges else 1.0
    )
    unauthorized_nodes, unauthorized_edges = path_scope_violations(path, principal)
    return {
        "case": case.case_id, "slice": case.slice, "terminal": terminal,
        "expected_terminal": case.expected_terminal, "predicted_relations": predicted,
        "relation_precision": precision, "relation_recall": recall,
        "path_exact": predicted == expected, "provenance_coverage": provenance,
        "unauthorized_nodes": unauthorized_nodes, "unauthorized_edges": unauthorized_edges,
        "nodes_visited": path.trace.nodes_visited, "edges_evaluated": path.trace.edges_evaluated,
    }


def run_text_case(case: PathEvalCase) -> dict:
    principal = Principal(f"text-{case.case_id}", case.tenant_id)
    hits = text_search(case.query, principal, k=2)
    hit_ids = {source.source_id for source in hits}
    expected_source_ids = {relation_by_id[rid].source_id for rid in case.expected_relation_ids}
    recall = len(hit_ids & expected_source_ids) / len(expected_source_ids) if expected_source_ids else 1.0
    return {"case": case.case_id, "text_source_ids": sorted(hit_ids), "text_evidence_recall": recall, "text_support_proxy": expected_source_ids.issubset(hit_ids)}


graph_df = pd.DataFrame([run_graph_case(case) for case in evaluation_cases])
text_df = pd.DataFrame([run_text_case(case) for case in evaluation_cases])
results_df = graph_df.merge(text_df, on="case")
display(results_df[["case", "slice", "terminal", "predicted_relations", "relation_recall", "text_evidence_recall"]])

assert (results_df.unauthorized_nodes == 0).all()
assert (results_df.unauthorized_edges == 0).all()
assert (results_df.provenance_coverage == 1.0).all()
assert (results_df.terminal == results_df.expected_terminal).all()


## 22. Measured path, provenance, authorization, and graph-vs-text outcomes

We aggregate only actual notebook outputs. Relation recall measures expected path relations retrieved divided by expected relations. Relation precision measures retrieved path relations that belong to the labelled path. Provenance coverage must be `1.0` for generation-eligible edges. Negative cases are scored by terminal reason rather than fluent prose.


In [ ]:
positive = results_df[results_df.predicted_relations.map(bool)]
overall_metrics = pd.Series({
    "mean_relation_precision": positive.relation_precision.mean(),
    "mean_relation_recall": positive.relation_recall.mean(),
    "exact_path_rate": results_df.path_exact.mean(),
    "terminal_accuracy": (results_df.terminal == results_df.expected_terminal).mean(),
    "provenance_coverage": results_df.provenance_coverage.mean(),
    "unauthorized_nodes_retrieved": int(results_df.unauthorized_nodes.sum()),
    "unauthorized_edges_retrieved": int(results_df.unauthorized_edges.sum()),
}, name="measured")
display(overall_metrics.to_frame())

slice_comparison = results_df.groupby("slice", as_index=False).agg(
    graph_relation_recall=("relation_recall", "mean"),
    graph_exact_path=("path_exact", "mean"),
    text_evidence_recall=("text_evidence_recall", "mean"),
    graph_edges_evaluated=("edges_evaluated", "mean"),
)
display(slice_comparison)

plot_df = slice_comparison.set_index("slice")[["graph_relation_recall", "text_evidence_recall"]]
ax = plot_df.plot.bar(figsize=(12, 4), color=["#16A3A5", "#2F6BFF"])
ax.set_ylim(0, 1.05)
ax.set_ylabel("mean recall")
ax.set_title("Measured graph-path relation recall vs text-source evidence recall")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

assert overall_metrics["provenance_coverage"] == 1.0
assert overall_metrics["unauthorized_nodes_retrieved"] == 0
assert overall_metrics["unauthorized_edges_retrieved"] == 0


## 23. No path is a diagnostic state, not a factual conclusion

`no_path_within_policy` can mean true absence, missed extraction, a false split, authorization exclusion, stale-edge filtering, or insufficient hop/fact budget. Record the stage and reason before deciding whether to retry extraction, expand budget, clarify the entity, or abstain.

| Failure | Observable signal | Appropriate response |
|---|---|---|
| extraction miss | gold/source relation absent from graph | re-extract or review source |
| false merge | fabricated path appears | split entity and rebuild affected edges |
| false split | expected path becomes unreachable | merge aliases with reviewed evidence |
| direction error | undirected path only | repair predicate direction/schema |
| path-budget failure | deeper authorized path exists | increase budget only if policy permits |
| hub explosion | many low-value facts evaluated | relation allowlist, weights, fact cap |
| authorization exclusion | unfiltered path exists; scoped path does not | abstain; never widen scope from query text |
| stale edge | historical relation answers current query | source-level reconciliation/version filter |
| source gap | no source-backed edge exists | retrieve text or report insufficient evidence |

> “GraphRAG failed” is not a sufficient diagnosis.


## 24. Local/global/DRIFT terminology and the technology boundary

This notebook implements **local relationship retrieval** with NetworkX. Microsoft GraphRAG is broader:

- **Local Search:** entity-focused graph context plus related raw text;
- **Global Search:** map-reduce over generated community reports for corpus-level questions;
- **DRIFT Search:** community-informed starting context plus iterative local exploration;
- **Basic Search:** a vector-RAG comparison path.

The notebook does not implement community reports or claim to reproduce Microsoft GraphRAG. `MultiDiGraph` is an inspectable teaching runtime, not production infrastructure. Moving to Neo4j or another graph store changes durability, transactions, indexes, query language, concurrency, access control, operations, backup, and scaling—not merely the import statement.

Graph extraction introduces substantial ingestion work because text units must be processed for entities, relationships, resolution, and optional summaries. Cost depends on chunking, batching, extraction strategy, model choice, caching, and incremental updates.


## 25. Production upgrade path

| Teaching implementation | Production upgrade |
|---|---|
| Frozen extraction fixture | versioned structured extraction, review queues, held-out extraction evals |
| Alias/type/tenant resolver | authoritative IDs, probabilistic candidates, human merge/split workflow |
| In-memory `MultiDiGraph` | graph database with transactions, indexes, backups, concurrency, query controls |
| Process-local traversal | service-enforced authorization, timeouts, cancellation, rate and cost budgets |
| Static relation schema | governed ontology versions and migration tooling |
| Source substring validation | immutable source locators, content hashes, extraction lineage |
| Current/historical flag | valid-time and transaction-time semantics, source reconciliation jobs |
| Deterministic renderer | calibrated synthesis model plus claim/citation verification |
| Local metrics | traces, query-slice dashboards, drift alerts, release gates, rollback |

**Security boundary:** do not rely on graph filtering after traversal. Candidate nodes, edges, properties, source spans, caches, logs, and community summaries all need scope-aware controls.


## Exercises

1. Add an allowed reverse transition for `LOCATED_IN` and answer “Which databases are in the EU?” without making every relation reversible.
2. Lower `max_facts` until a valid multi-hop case fails; classify it as a budget failure rather than factual absence.
3. Add a relation with a valid schema but a low-authority source; decide whether it can seed retrieval, support generation, or only trigger review.
4. Implement pairwise precision/recall over a larger entity-resolution gold set.
5. Add valid-time intervals and answer the same dependency question for 2024 and 2026.
6. Replace BM25 seeding with an embedding retriever and compare seed recall without changing traversal policy.
7. Add a tiny deterministic community partition and explain why it is not Microsoft GraphRAG's full global pipeline.
8. Persist the graph in Neo4j and document which authorization invariant moved to the storage layer.

## Summary

You evolved a three-edge connectivity demo into a source-backed graph-retrieval system. The lab extracts or loads typed facts, validates them, resolves entities, preserves parallel-edge provenance, enforces directional and tenant-aware traversal, controls fan-out, hydrates source spans, generates cited output, injects failures, and compares graph paths with text retrieval on a shared labelled set.

**Deliberately deferred:** full Microsoft GraphRAG community indexing/query, production graph databases, distributed extraction, learned entity resolution, unrestricted live web retrieval, and production authorization infrastructure.
